装饰器，我们现在讨论如何同时实现有参和无参装饰器的实现。

In [ ]:
import functools

# 你看不懂的这行，是整个兼容装饰器的核心
def log(func=None, *, text='call'):
    # 带参调用场景：func是None，返回一个接收函数的lambda
    if func is None:
        return lambda f: log(f, text=text)
    
    # 无参调用场景：func是被装饰的函数，直接走装饰逻辑
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f'begin call {func.__name__}')
        print(f'{text} {func.__name__}')
        result = func(*args, **kwargs)
        print(f'end call {func.__name__}\n')
        return result
    
    return wrapper

# 测试1：无参写法
@log
def test1():
    print("执行test1函数")

# 测试2：带参写法
@log(text='execute')
def test2(a, b):
    print(f"执行test2函数，计算{a}+{b}")
    return a + b

# 运行测试
test1()
test2(1, 2)


def log(func=None, *, text='call'):
    if func is None:
        return lambda f: log(f, text=text)
主要是理解这个是什么意思。

那么他是如何同时实现无参和有参decorator的呢？

先明白 * 的作用，在函数参数列表中， * 后面的参数必须是关键字参数，即你不能只写"call",这样就成了位置参数，直接报错。看后面就明白了。

从例子中看：
无参log:@log 相当于log(test1)，而有参相当于lambda(test2)，怎么来的呢？第一次log(text='execute')会进入if语句，里面返回一个lambda,这个lambda要接收一个参数给f，于是lambda(test2)就执行了。
这里是一个函数的链式调用，对于有参log就先执行第一个括号的内容，func为None进入if最后返回一个lambda,再执行第二个括号内的，这里func就被赋值为test2了，剩下的就是正常进行了。

当我们学习了偏函数之后，就可以用partial代替这里的lambda了，可读性更强，原理相同：

In [ ]:
import functools

# 你看不懂的这行，是整个兼容装饰器的核心
def log(func=None, *, text='call'):
    # 带参调用场景：func是None，返回一个接收函数的lambda
    if func is None:
        return functools.partial(log,text=text)
    
    # 无参调用场景：func是被装饰的函数，直接走装饰逻辑
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f'begin call {func.__name__}')
        print(f'{text} {func.__name__}')
        result = func(*args, **kwargs)
        print(f'end call {func.__name__}\n')
        return result
    
    return wrapper

# 测试1：无参写法
@log
def test1():
    print("执行test1函数")

# 测试2：带参写法
@log(text='execute')
def test2(a, b):
    print(f"执行test2函数，计算{a}+{b}")
    return a + b

# 运行测试
test1()
test2(1,2)

_____________________________________________________________________________________________________________________________

In [ ]:
import time,functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*arg,**kw):
        start=time.perf_counter()
        res=func(*arg,**kw)
        end=time.perf_counter()
        differ=end-start
        print("The time difference is ",differ)
        return res
    return wrapper

@timer
def slow(x,y):
    time.sleep(0.4)
    return x+y

slow(11,33)

用一个变量接收函数，而不是return 函数